In [1]:
import os
import tempfile
import json
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from unstructured.partition.xlsx import partition_xlsx
from unstructured.documents.elements import Table
from html_table_rescuer import TableParser
from bs4 import BeautifulSoup, Tag

filename = "test1.xlsx"
OUTPUT_JSON = "hasil_parsing_full.json"


# ============================================================
# 0. PARTISI PER SHEET (via file temporary)
# ============================================================
def partition_single_sheet(filepath: str, sheet_name: str):
    """Partisi HANYA satu sheet dengan membuat file XLSX sementara."""
    wb = load_workbook(filepath, data_only=True)
    for s in list(wb.sheetnames):
        if s != sheet_name:
            del wb[s]

    tmp_fd, tmp_path = tempfile.mkstemp(suffix=".xlsx")
    os.close(tmp_fd)
    try:
        wb.save(tmp_path)
        elements = partition_xlsx(
            filename=tmp_path,
            infer_table_structure=True,
            find_subtable=True,
        )
    finally:
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)

    return elements


# ============================================================
# 0a. PROMOSI BARIS PERTAMA MENJADI <th>
# ============================================================
def promote_first_row_to_header(html: str) -> str:
    """Ubah baris pertama tabel dari <td> menjadi <th> dan bungkus <thead>."""
    if not html:
        return html
    try:
        soup = BeautifulSoup(html, "lxml")
        table = soup.find("table")
        if not table:
            return html

        if table.find("thead") or table.find("th"):
            return html

        rows = table.find_all("tr")
        if not rows:
            return html

        first_row = rows[0]
        cells = first_row.find_all("td")
        if not cells:
            return html

        thead = soup.new_tag("thead")
        new_tr = soup.new_tag("tr")
        for cell in cells:
            th = soup.new_tag("th")
            th.string = cell.get_text(strip=True)
            new_tr.append(th)
        thead.append(new_tr)

        first_row.extract()
        table.insert(0, thead)

        return str(table)
    except Exception:
        return html


# ============================================================
# 0b. KONVERSI HTML TABEL -> MARKDOWN
# ============================================================
def html_table_to_markdown(html: str, strategy: str = "fill_dito") -> str | None:
    """Konversi HTML tabel dari unstructured.metadata.text_as_html menjadi Markdown."""
    if not html:
        return None
    try:
        html_with_header = promote_first_row_to_header(html)
        parser = TableParser(html_with_header)
        tables = parser.parse()
        if not tables:
            return None
        md_tables = []
        for t in tables:
            try:
                md_tables.append(t.to_markdown())
            except Exception:
                continue
        if not md_tables:
            return None
        return "\n\n".join(md_tables)
    except Exception:
        return None


# ============================================================
# 1. EKSTRAKSI GAMBAR & HYPERLINK DENGAN POSISI BARIS
# ============================================================
def extract_sheet_assets(filepath):
    """Return {sheet_name: {"images": [...], "hyperlinks": [...]}} dengan row/col."""
    wb = load_workbook(filepath, data_only=False)
    img_dir = Path("extracted_images")
    img_dir.mkdir(exist_ok=True)
    result = {}

    for ws in wb.worksheets:
        images = []
        hyperlinks = []

        # --- Gambar ---
        for img in ws._images:
            try:
                row = img.anchor._from.row + 1
                col = img.anchor._from.col + 1
                img_data = img._data()
                ext = "png"
                if getattr(img, "format", None):
                    ext = img.format.lower()
                img_filename = f"{ws.title.replace(' ', '_')}_r{row}c{col}.{ext}"
                img_path = img_dir / img_filename
                img_path.write_bytes(img_data)

                # Cari label konteks: nilai terdekat di kiri, lalu di atas
                label = None
                for c in range(col, 0, -1):
                    v = ws.cell(row=row, column=c).value
                    if v and str(v).strip():
                        label = str(v).strip()
                        break
                if not label:
                    for r in range(row, 0, -1):
                        v = ws.cell(row=r, column=col).value
                        if v and str(v).strip():
                            label = str(v).strip()
                            break

                images.append({
                    "row": row,
                    "col": col,
                    "cell": f"{get_column_letter(col)}{row}",
                    "label": label,
                    "path": str(img_path),
                })
            except Exception as e:
                images.append({"error": str(e), "row": 0, "col": 0})

        # --- Hyperlink ---
        for row in ws.iter_rows():
            for cell in row:
                if cell.hyperlink:
                    hyperlinks.append({
                        "row": cell.row,
                        "col": cell.column,
                        "cell": cell.coordinate,
                        "text": str(cell.value) if cell.value else "",
                        "target": cell.hyperlink.target,
                        "tooltip": cell.hyperlink.tooltip,
                    })

        images.sort(key=lambda x: (x.get("row", 0), x.get("col", 0)))
        hyperlinks.sort(key=lambda x: (x["row"], x["col"]))
        result[ws.title] = {"images": images, "hyperlinks": hyperlinks}

    return result


# ============================================================
# 2. CARI BARIS AWAL & AKHIR ELEMEN DI SHEET
# ============================================================
def element_start_row(ws, element_text, start_row=1):
    """Cari baris awal elemen berdasarkan token unik dari teksnya."""
    if not element_text:
        return start_row
    snippet = element_text.strip()[:80]
    tokens = [t for t in snippet.split() if len(t) > 2][:3]
    if not tokens:
        return start_row

    for r in range(start_row, ws.max_row + 1):
        row_text = " ".join(
            str(ws.cell(row=r, column=c).value or "")
            for c in range(1, ws.max_column + 1)
        )
        if all(tok in row_text for tok in tokens):
            return r
    return start_row


def element_end_row(el, start_row):
    """Untuk Table, hitung jumlah <tr>; untuk lainnya, 1 baris."""
    if not isinstance(el, Table):
        return start_row
    html = getattr(el.metadata, "text_as_html", "") or ""
    tr_count = html.count("<tr>")
    return start_row + max(tr_count - 1, 0)


# ============================================================
# 3. MAIN
# ============================================================
assets = extract_sheet_assets(filename)

wb = load_workbook(filename, data_only=True)
sheet_names = wb.sheetnames

output = []

for sheet_name in sheet_names:
    elements = partition_single_sheet(filename, sheet_name)

    ws = wb[sheet_name]
    sheet_assets = assets.get(sheet_name, {"images": [], "hyperlinks": []})

    # Gabung semua aset, urutkan berdasarkan (row, col)
    all_assets = []
    for img in sheet_assets["images"]:
        if "error" in img:
            continue
        all_assets.append({
            "type": "image",
            "row": img["row"],
            "col": img["col"],
            "cell": img["cell"],
            "path": str(Path(img["path"]).resolve()),
            "label": img.get("label"),
        })
    for link in sheet_assets["hyperlinks"]:
        all_assets.append({
            "type": "link",
            "row": link["row"],
            "col": link["col"],
            "cell": link["cell"],
            "target": link["target"],
            "text": link["text"],
            "tooltip": link.get("tooltip"),
        })
    all_assets.sort(key=lambda x: (x["row"], x["col"]))

    # DEDUP: URL yang sudah dirender sebagai aset
    asset_urls = set()
    for link in sheet_assets["hyperlinks"]:
        if link.get("target"):
            asset_urls.add(link["target"].strip())
        if link.get("text"):
            asset_urls.add(link["text"].strip())

    def is_asset_url(text: str) -> bool:
        if not text:
            return False
        return text.strip() in asset_urls

    # Tentukan rentang baris setiap elemen
    element_spans = []
    cursor = 1
    for el in elements:
        start = element_start_row(ws, el.text, cursor)
        end = element_end_row(el, start)
        element_spans.append((start, end))
        cursor = end + 1

    emitted = set()

    def emit_assets_until(max_row):
        """Emit aset yang row-nya <= max_row dan belum pernah di-emit."""
        for a in all_assets:
            key = (a["row"], a["col"], a["type"])
            if key in emitted:
                continue
            if a["row"] <= max_row:
                if a["type"] == "image":
                    output.append({
                        "type": "image",
                        "img_path": a["path"],
                        "image_caption": [],
                        "image_footnote": [],
                        "page_idx": sheet_name,
                    })
                elif a["type"] == "link":
                    # ✅ Link ditulis sebagai text block
                    link_text = (a["text"] or "").strip()
                    target = (a["target"] or "").strip()
                    if link_text and target and link_text != target:
                        text_value = f"{link_text} ({target})"
                    else:
                        text_value = target or link_text
                    if text_value:
                        output.append({
                            "type": "text",
                            "text": text_value,
                            "page_idx": sheet_name,
                        })
                emitted.add(key)

    for idx, el in enumerate(elements):
        start_row, end_row = element_spans[idx]
        emit_assets_until(end_row)

        # SKIP: elemen non-Table yang teksnya adalah URL aset
        if not isinstance(el, Table) and is_asset_url(el.text):
            continue

        if isinstance(el, Table):
            html = getattr(el.metadata, "text_as_html", None)
            md_table = html_table_to_markdown(html) if html else None

            table_body = md_table if md_table else (html or "")
            output.append({
                "type": "table",
                "table_body": table_body,
                "table_caption": [],
                "table_footnote": [],
                "page_idx": sheet_name,
            })
        else:
            text = (el.text or "").strip()
            if text:
                output.append({
                    "type": "text",
                    "text": text,
                    "page_idx": sheet_name,
                })

    # Sisa aset yang row-nya setelah elemen terakhir
    emit_assets_until(10**9)

# ============================================================
# 4. SIMPAN JSON
# ============================================================
output_path = Path(OUTPUT_JSON)
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Selesai: {output_path.resolve()}")
print(f"Total elemen: {len(output)}")
print(f"Gambar disimpan di: {Path('extracted_images').resolve()}")

Profile format error in "/home/satyananda/hybrid-rag/.venv/lib/python3.12/site-packages/langdetect/profiles/es"


Selesai: /home/satyananda/hybrid-rag/examples/hasil_parsing_full.json
Total elemen: 68
Gambar disimpan di: /home/satyananda/hybrid-rag/examples/extracted_images


In [10]:
import os
import json
from pathlib import Path
from docx import Document
from docx.oxml.ns import qn
from docx.table import Table as DocxTable
from docx.text.paragraph import Paragraph
from unstructured.partition.docx import partition_docx
from unstructured.documents.elements import Table as UnstructuredTable, Image as UnstructuredImage
from html_table_rescuer import TableParser


# ============================================================
# KONFIGURASI
# ============================================================
DOCX_PATH = "test1.docx"
IMAGE_OUTPUT_DIR = "./docx_images"
OUTPUT_JSON = "output.json"

os.makedirs(IMAGE_OUTPUT_DIR, exist_ok=True)


# ============================================================
# NAMESPACE
# ============================================================
W_NS = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"
A_NS = "http://schemas.openxmlformats.org/drawingml/2006/main"
M_NS = "http://schemas.openxmlformats.org/officeDocument/2006/math"


# ============================================================
# FOOTNOTE / ENDNOTE
# ============================================================
def get_footnotes_map(doc):
    footnotes = {}
    try:
        part = doc.part.package.part_related_by(
            "http://schemas.openxmlformats.org/officeDocument/2006/relationships/footnotes"
        )
        root = part.element
        ns = {"w": W_NS}
        for fn in root.findall(".//w:footnote", ns):
            fn_id = fn.get(qn("w:id"))
            if fn_id in ("-1", "0"):
                continue
            texts = [t.text or "" for t in fn.findall(".//w:t", ns)]
            footnotes[fn_id] = " ".join(texts).strip()
    except Exception:
        pass
    return footnotes


def get_endnotes_map(doc):
    endnotes = {}
    try:
        part = doc.part.package.part_related_by(
            "http://schemas.openxmlformats.org/officeDocument/2006/relationships/endnotes"
        )
        root = part.element
        ns = {"w": W_NS}
        for en in root.findall(".//w:endnote", ns):
            en_id = en.get(qn("w:id"))
            if en_id in ("-1", "0"):
                continue
            texts = [t.text or "" for t in en.findall(".//w:t", ns)]
            endnotes[en_id] = " ".join(texts).strip()
    except Exception:
        pass
    return endnotes


def extract_footnote_refs(paragraph):
    refs = []
    for ref in paragraph._element.findall(f".//{{{W_NS}}}footnoteReference"):
        refs.append(("footnote", ref.get(qn("w:id"))))
    for ref in paragraph._element.findall(f".//{{{W_NS}}}endnoteReference"):
        refs.append(("endnote", ref.get(qn("w:id"))))
    return refs


# ============================================================
# RUMUS OMML
# ============================================================
def extract_omml_equations(paragraph):
    equations = []
    for omath in paragraph._element.findall(f".//{{{M_NS}}}oMath"):
        texts = [t.text or "" for t in omath.findall(f".//{{{M_NS}}}t")]
        equations.append({
            "latex": "",
            "text": " ".join(texts).strip(),
            "is_display": False,
        })
    for omathpara in paragraph._element.findall(f".//{{{M_NS}}}oMathPara"):
        for omath in omathpara.findall(f".//{{{M_NS}}}oMath"):
            texts = [t.text or "" for t in omath.findall(f".//{{{M_NS}}}t")]
            equations.append({
                "latex": "",
                "text": " ".join(texts).strip(),
                "is_display": True,
            })
    return equations


# ============================================================
# GAMBAR (inline + floating)
# ============================================================
def extract_images_from_paragraph(paragraph, doc, img_dir):
    images = []
    for drawing in paragraph._element.findall(f".//{{{W_NS}}}drawing"):
        for blip in drawing.findall(f".//{{{A_NS}}}blip"):
            rId = blip.get(qn("r:embed"))
            if rId is None:
                continue
            try:
                image_part = doc.part.related_parts[rId]
                ext = image_part.partname.ext
                filename = f"img_{rId}.{ext}"
                path = Path(img_dir) / filename
                path.write_bytes(image_part.blob)
                images.append({"img_path": str(path.resolve()), "rId": rId})
            except Exception:
                continue
    return images


def has_page_break(paragraph):
    for br in paragraph._element.findall(f".//{{{W_NS}}}br"):
        if br.get(qn("w:type")) == "page":
            return True
    return False


# ============================================================
# DETEKSI CAPTION — STYLE-BASED, TANPA REGEX
# ============================================================
def is_caption_paragraph(paragraph) -> bool:
    """True jika paragraph memakai style yang mengandung kata 'caption'.

    Fleksibel untuk 'Caption', 'Caption2', 'Figure Caption', dll.
    Tidak bergantung pada isi teks, sehingga aman untuk berbagai penamaan.
    """
    try:
        style = paragraph.style
        if style is None:
            return False
        return "caption" in style.name.lower()
    except Exception:
        return False


# ============================================================
# KONVERSI TABEL HTML -> MARKDOWN
# ============================================================
def html_table_to_markdown(html: str) -> str | None:
    if not html:
        return None
    try:
        parser = TableParser(html)
        tables = parser.parse()
        if not tables:
            return None
        return "\n\n".join(t.to_markdown() for t in tables)
    except Exception:
        return None


def docx_table_to_html(table: DocxTable) -> str:
    rows = []
    for row in table.rows:
        cells = []
        for cell in row.cells:
            text = " ".join(p.text for p in cell.paragraphs).strip()
            cells.append(f"<td>{text}</td>")
        rows.append("<tr>" + "".join(cells) + "</tr>")
    return "<table>" + "".join(rows) + "</table>"


# ============================================================
# ITERASI BLOCK
# ============================================================
def iter_block_items(doc):
    body = doc.element.body
    for child in body.iterchildren():
        if child.tag == qn("w:p"):
            yield Paragraph(child, doc)
        elif child.tag == qn("w:tbl"):
            yield DocxTable(child, doc)


def classify_block(block, doc):
    """Klasifikasikan satu block menjadi tuple (type, payload).

    Type:
      - "image"    : payload = list of {img_path, rId}
      - "table"    : payload = DocxTable
      - "equation" : payload = list of {latex, text, is_display}
      - "caption"  : payload = str  (style-based, tanpa regex)
      - "text"     : payload = str
      - "empty"    : payload = None
    """
    if isinstance(block, DocxTable):
        return ("table", block)

    if not isinstance(block, Paragraph):
        return ("empty", None)

    # 1. Gambar
    imgs = extract_images_from_paragraph(block, doc, IMAGE_OUTPUT_DIR)
    if imgs:
        return ("image", imgs)

    # 2. Rumus
    eqs = extract_omml_equations(block)
    if eqs:
        return ("equation", eqs)

    # 3. Teks / Caption
    txt = (block.text or "").strip()
    if not txt:
        return ("empty", None)

    if is_caption_paragraph(block):
        return ("caption", txt)

    return ("text", txt)


# ============================================================
# CAPTION MATCHING — AMBIL SATU CAPTION TERAKHIR
# ============================================================
def find_caption_for_element(classified, idx, consumed):
    """Ambil SATU caption untuk elemen di `idx`.

    Strategi:
      - Kumpulkan semua caption berurutan yang berdekatan (skip empty).
      - Hentikan walk saat menemukan non-caption.
      - Tandai SEMUA caption yang terkumpul sebagai `consumed`
        (agar tidak muncul sebagai text).
      - Attach HANYA caption terakhir (paling deskriptif, biasanya
        'Gambar N: ...' atau 'Tabel N: ...').

    Return: list of caption string (0 atau 1 elemen).
    """
    candidates = []   # list of (index, text) sesuai urutan dokumen

    # --- Walk BACKWARD ---
    j = idx - 1
    while j >= 0:
        if j in consumed:
            j -= 1
            continue
        t, p = classified[j]
        if t == "empty":
            j -= 1
            continue
        if t == "caption":
            candidates.insert(0, (j, p))
            j -= 1
        else:
            break

    # --- Walk FORWARD ---
    j = idx + 1
    while j < len(classified):
        if j in consumed:
            j += 1
            continue
        t, p = classified[j]
        if t == "empty":
            j += 1
            continue
        if t == "caption":
            candidates.append((j, p))
            j += 1
        else:
            break

    if not candidates:
        return []

    # Tandai semua caption yang terkumpul sebagai consumed
    for ci, _ in candidates:
        consumed.add(ci)

    # Attach HANYA caption terakhir
    return [candidates[-1][1]]


# ============================================================
# MAP TABEL DARI UNSTRUCTURED
# ============================================================
def build_unstructured_table_map(elements):
    mapping = {}
    idx = 0
    for el in elements:
        if isinstance(el, UnstructuredTable):
            html = getattr(el.metadata, "text_as_html", None)
            if html:
                mapping[idx] = html
            idx += 1
    return mapping


# ============================================================
# MAIN
# ============================================================
def main():
    doc = Document(DOCX_PATH)

    # 1. Unstructured untuk tabel HTML
    us_elements = partition_docx(
        filename=DOCX_PATH,
        infer_table_structure=True,
        extract_images=True,
        image_output_dir_path=IMAGE_OUTPUT_DIR,
    )
    us_table_html_map = build_unstructured_table_map(us_elements)

    # 2. Footnote & endnote
    footnotes_map = get_footnotes_map(doc)
    endnotes_map = get_endnotes_map(doc)

    # 3. Klasifikasi semua block
    blocks = list(iter_block_items(doc))
    classified = [classify_block(b, doc) for b in blocks]

    # 4. Iterasi
    output = []
    consumed = set()
    page_idx = 0
    table_counter = 0

    for idx, (btype, payload) in enumerate(classified):
        # Update page_idx kalau ada page break
        if isinstance(blocks[idx], Paragraph) and has_page_break(blocks[idx]):
            page_idx += 1

        if idx in consumed:
            continue

        # ----- IMAGE -----
        if btype == "image":
            caption = find_caption_for_element(classified, idx, consumed)
            for img in payload:
                output.append({
                    "type": "image",
                    "img_path": img["img_path"],
                    "image_caption": caption,
                    "image_footnote": [],
                    "page_idx": page_idx,
                })

        # ----- TABLE -----
        elif btype == "table":
            html = us_table_html_map.get(table_counter)
            if not html:
                html = docx_table_to_html(payload)
            md_table = html_table_to_markdown(html) or html

            caption = find_caption_for_element(classified, idx, consumed)

            output.append({
                "type": "table",
                "table_body": md_table,
                "table_caption": caption,
                "table_footnote": [],
                "page_idx": page_idx,
            })
            table_counter += 1

        # ----- EQUATION -----
        elif btype == "equation":
            for eq in payload:
                output.append({
                    "type": "equation",
                    "latex": eq["latex"],
                    "text": eq["text"],
                    "page_idx": page_idx,
                })

        # ----- TEXT -----
        elif btype == "text":
            refs = extract_footnote_refs(blocks[idx])
            fn_texts = []
            for kind, fid in refs:
                if kind == "footnote" and fid in footnotes_map:
                    fn_texts.append(footnotes_map[fid])
                elif kind == "endnote" and fid in endnotes_map:
                    fn_texts.append(endnotes_map[fid])

            item = {"type": "text", "text": payload, "page_idx": page_idx}
            if fn_texts:
                item["footnote"] = fn_texts
            output.append(item)

        # ----- CAPTION ORPHAN (tidak terpakai) -----
        elif btype == "caption":
            output.append({
                "type": "text",
                "text": payload,
                "page_idx": page_idx,
            })

        # ----- EMPTY: skip -----

    # 5. Tulis output
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

    print(f"Selesai: {OUTPUT_JSON} ({len(output)} elemen)")
    print(f"Gambar disimpan di: {Path(IMAGE_OUTPUT_DIR).resolve()}")


if __name__ == "__main__":
    main()

Selesai: output.json (19 elemen)
Gambar disimpan di: /home/satyananda/hybrid-rag/notebooks/docx_images
